# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is provided via a Croissant schema URL published by SenScience.

- **Dataset Title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **Schema URL:** https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Let's load the metadata and records provided by the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review the available record sets and examine their fields and ID references (`@id`). All entities, including record sets and fields, are referenced by their `@id` as per Croissant schema conventions.

Below, we print all record set `@id`s, then for each, all available fields and their `@id` and `name`.

In [ ]:
# List all record sets and their fields

record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"Record set name: {rs.name} | @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("Fields (@id, name):")
    for field in rs.fields:
        print(f"  - {field.id} : {field.name} | type: {field.data_type}")
    print('---')

# Example: Print a few records from each record set (by @id)
for rs in record_set_ids:
    print(f"\nExample record from {rs}:")
    for i, rec in enumerate(dataset.records(record_set=rs)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for programmatic analysis.

*All accesses to fields, record sets, and columns are by their `@id` as per FAIR and Croissant guidelines.*

In [ ]:
# Extract all record sets into dataframes using their @id
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record set {rs_id} with shape {df.shape}")

# For further analysis, pick first non-empty record set
main_rs_id = None
for rs_id in record_set_ids:
    if len(dataframes[rs_id]) > 0:
        main_rs_id = rs_id
        break

if main_rs_id:
    print(f"\nColumns in main DataFrame ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No records found in the record sets.")

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set. All field references are made by their `@id`.

- We'll select a numeric column (`@id`) for filtering and normalization.
- We'll group by a categorical field (if present).

*Note: If the dataset does not contain records, this step will be skipped.*

In [ ]:
import numpy as np

if main_rs_id is not None:
    df = dataframes[main_rs_id].copy()

    # Print numeric columns using dtypes
    numeric_cols = df.select_dtypes(include=['number', np.number, 'float', 'int']).columns.tolist()
    if not numeric_cols:
        # Try to auto-convert columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_cols = df.select_dtypes(include=['number', np.number, 'float', 'int']).columns.tolist()

    print(f"Numeric fields detected (by @id): {numeric_cols}")

    # Pick the first numeric field for analysis
    if numeric_cols:
        numeric_field_id = numeric_cols[0]

        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)

        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Determine a potential groupby field (categorical string column)
        group_field = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != numeric_field_id:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped data by '{group_field}' (mean of '{numeric_field_id}'):")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields available for EDA in the main record set.")
else:
    print("No records available for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and, where possible, show group-wise distributions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if main_rs_id is not None and dataframes[main_rs_id].shape[0] > 0 and numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {main_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by group_field if available
    if group_field is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=dataframes[main_rs_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load and explore a complex scientific dataset described by the Croissant schema. All field, column, and record set references were by `@id` for traceability and reproducibility.

- Loaded dataset metadata and records programmatically.
- Explored available record sets and fields, referencing them by `@id`.
- Extracted a main record set to a pandas DataFrame, performed numeric filtering and normalization, and grouped data by attributes.
- Visualized the distribution of a selected numeric attribute.

This approach can be generalized to any Croissant-described dataset, ensuring full FAIR and schema-compliant data handling for analytics and machine learning workflows.